# Oil News Project — Demo Notebook

End-to-end walkthrough of the Brent crude oil price prediction pipeline:

1. Database configuration
2. Load datasets into MySQL
3. Train the Ridge regression model
4. Generate a multi-day forward forecast
5. Visualize actual vs predicted prices

> **Run cells top-to-bottom.** MySQL must be reachable for step 2; steps 3–5 only need the CSV datasets.

## Setup
Ensures the notebook's working directory matches the project root and that `src/` is on the import path — mirroring the environment the standalone scripts assume.

In [ ]:
import sys, os
from pathlib import Path

# Ensure the notebook CWD is the project root so relative paths work
# (same assumption the standalone scripts make when run from the root).
_root = Path.cwd()
if (_root / "src").exists():
    # Already at project root
    pass
elif (_root.parent / "src").exists():
    os.chdir(_root.parent)
    _root = Path.cwd()

# Make src/ importable (mirrors the sys.path guard in load_mysql.py)
_src = str(_root / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

print(f"Project root: {Path.cwd()}")

## 1. Database Configuration
Loads MySQL connection details from `.env` (or environment variables). Source: `src/db_config.py`

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class MySQLConfig:
    host: str
    port: int
    user: str
    password: str
    database: str


def load_dotenv(path: Path = Path(".env")) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def get_mysql_config() -> MySQLConfig:
    load_dotenv()
    return MySQLConfig(
        host=os.getenv("MYSQL_HOST", "127.0.0.1"),
        port=int(os.getenv("MYSQL_PORT", "3306")),
        user=os.getenv("MYSQL_USER", "root"),
        password=os.getenv("MYSQL_PASSWORD", ""),
        database=os.getenv("MYSQL_DATABASE", "oil_news_project"),
    )

## 2. Load Datasets into MySQL
Reads every CSV in `datasets/` and upserts it into MySQL, then applies the analytics views SQL. Source: `src/load_mysql.py`

In [ ]:
from __future__ import annotations

import argparse
import csv
import re
from collections import defaultdict
from pathlib import Path
from typing import Iterable


from db_config import get_mysql_config


DATASET_DIR = Path("datasets")

DATE_COLUMNS = {
    "trade_date",
    "event_date",
    "gpr_date",
    "month_start",
    "snapshot_date",
    "market_date",
    "full_date",
}


def require_connector():
    try:
        import mysql.connector  # type: ignore
    except ModuleNotFoundError as exc:
        raise SystemExit(
            "mysql-connector-python is required for loading MySQL. "
            "Install with: python -m pip install -r requirements.txt"
        ) from exc
    return mysql.connector


def q(identifier: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Unsafe identifier: {identifier}")
    return f"`{identifier}`"


def load_data_dictionary(path: Path) -> dict[str, dict[str, str]]:
    if not path.exists():
        return {}
    mapping: dict[str, dict[str, str]] = defaultdict(dict)
    with path.open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            mapping[row["table_name"]][row["column_name"]] = row["dtype"]
    return mapping


def mysql_type(column: str, dtype: str | None) -> str:
    if column in DATE_COLUMNS or (dtype and "datetime" in dtype):
        return "DATE"
    if dtype and "int" in dtype:
        return "INT"
    if dtype and "float" in dtype:
        return "DOUBLE"
    if column.endswith("_description") or column in {"description", "policy_response"}:
        return "TEXT"
    return "VARCHAR(512)"


def primary_key_for(table: str, columns: list[str]) -> str | None:
    candidates = [column for column in columns if column.endswith("_id") or column.endswith("_key")]
    if candidates and candidates[0] in columns:
        return candidates[0]
    if table.startswith("ops_"):
        candidate = table.removeprefix("ops_").rstrip("s") + "_id"
        return candidate if candidate in columns else None
    return None


def create_table_sql(table: str, columns: list[str], dtypes: dict[str, str]) -> str:
    pk = primary_key_for(table, columns)
    definitions = []
    for column in columns:
        col_type = mysql_type(column, dtypes.get(column))
        nullable = "NOT NULL" if column == pk else "NULL"
        definitions.append(f"  {q(column)} {col_type} {nullable}")
    if pk:
        definitions.append(f"  PRIMARY KEY ({q(pk)})")
    return f"CREATE TABLE IF NOT EXISTS {q(table)} (\n" + ",\n".join(definitions) + "\n) ENGINE=InnoDB;"


def iter_csv_rows(path: Path, columns: list[str]) -> Iterable[tuple[object, ...]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            values: list[object] = []
            for column in columns:
                value = row.get(column, "")
                values.append(None if value == "" else value)
            yield tuple(values)


def load_csv(cursor, table: str, path: Path, dtypes: dict[str, str], replace: bool) -> int:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        columns = list(next(csv.reader(handle)))

    cursor.execute(create_table_sql(table, columns, dtypes))
    if replace:
        cursor.execute(f"TRUNCATE TABLE {q(table)}")

    placeholders = ", ".join(["%s"] * len(columns))
    column_sql = ", ".join(q(column) for column in columns)
    insert_sql = f"INSERT INTO {q(table)} ({column_sql}) VALUES ({placeholders})"

    batch: list[tuple[object, ...]] = []
    total = 0
    for values in iter_csv_rows(path, columns):
        batch.append(values)
        if len(batch) >= 1000:
            cursor.executemany(insert_sql, batch)
            total += len(batch)
            batch.clear()
    if batch:
        cursor.executemany(insert_sql, batch)
        total += len(batch)
    return total


def apply_sql_file(cursor, path: Path, database: str) -> None:
    if not path.exists():
        return
    sql = path.read_text(encoding="utf-8").replace("oil_news_project", database)
    for statement in [part.strip() for part in sql.split(";") if part.strip()]:
        cursor.execute(statement)


def main() -> None:
    parser = argparse.ArgumentParser(description="Load workspace CSV datasets into MySQL.")
    parser.add_argument("--dataset-dir", type=Path, default=DATASET_DIR)
    parser.add_argument("--replace", action="store_true", help="Truncate tables before loading.")
    parser.add_argument("--only", nargs="*", help="Optional list of CSV stem/table names to load.")
    args = parser.parse_args([])

    mysql = require_connector()
    config = get_mysql_config()
    connection = mysql.connect(
        host=config.host,
        port=config.port,
        user=config.user,
        password=config.password,
        autocommit=False,
    )
    cursor = connection.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {q(config.database)}")
    cursor.execute(f"USE {q(config.database)}")

    dictionary = load_data_dictionary(args.dataset_dir / "data_dictionary.csv")
    csv_files = sorted(args.dataset_dir.glob("*.csv"))
    if args.only:
        wanted = set(args.only)
        csv_files = [path for path in csv_files if path.stem in wanted]

    for csv_path in csv_files:
        table = csv_path.stem
        loaded = load_csv(cursor, table, csv_path, dictionary.get(table, {}), args.replace)
        connection.commit()
        print(f"Loaded {loaded:>6} rows into {table}")

    apply_sql_file(cursor, Path("sql") / "analytics_views.sql", config.database)
    connection.commit()
    cursor.close()
    connection.close()
    print(f"Done. Database `{config.database}` is ready.")

In [ ]:
# Execute: load all CSVs into MySQL
main()

## 3. Train Oil Price Model
Builds feature/label pairs with a configurable horizon, trains a `StandardScaler → Ridge` pipeline, and saves the model + metrics. Source: `src/train_oil_model.py`

In [ ]:
from __future__ import annotations

import argparse
import csv
import json
from datetime import date
from pathlib import Path

import joblib
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


FEATURE_COLUMNS = [
    "brent_price_usd",
    "wti_price_usd",
    "dxy_index",
    "vix_index",
    "gpr_index",
    "brent_return",
    "wti_return",
    "brent_lag_1",
    "brent_lag_3",
    "brent_lag_7",
    "wti_lag_1",
    "wti_lag_3",
    "wti_lag_7",
    "brent_volatility_7d",
    "brent_volatility_30d",
    "wti_volatility_7d",
    "wti_volatility_30d",
    "brent_wti_spread",
    "event_severity",
    "event_flag",
]


def to_float(value: str | None) -> float | None:
    if value is None or value == "":
        return None
    try:
        result = float(value)
    except ValueError:
        return None
    return result


def read_market_rows(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    return sorted(rows, key=lambda row: row["market_date"])


def build_examples(
    rows: list[dict[str, str]], horizon: int = 10
) -> tuple[list[list[float]], list[float], list[str]]:
    """Build feature/label pairs where the target is `horizon` trading days ahead."""
    x_rows: list[list[float]] = []
    y_rows: list[float] = []
    dates: list[str] = []
    for idx, row in enumerate(rows[: len(rows) - horizon]):
        future_brent = to_float(rows[idx + horizon].get("brent_price_usd"))
        features = [to_float(row.get(column)) for column in FEATURE_COLUMNS]
        if future_brent is None or any(value is None for value in features):
            continue
        x_rows.append([float(value) for value in features])
        y_rows.append(future_brent)
        dates.append(rows[idx + horizon]["market_date"])
    return x_rows, y_rows, dates


def split_chronological(
    x_rows: list[list[float]], y_rows: list[float], dates: list[str], test_ratio: float
) -> tuple[list[list[float]], list[float], list[str], list[list[float]], list[float], list[str]]:
    split_index = max(1, int(len(x_rows) * (1.0 - test_ratio)))
    return (
        x_rows[:split_index],
        y_rows[:split_index],
        dates[:split_index],
        x_rows[split_index:],
        y_rows[split_index:],
        dates[split_index:],
    )


def metrics(actual: list[float], predicted: list[float]) -> dict[str, float]:
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": mean_squared_error(actual, predicted) ** 0.5,
        "mape_pct": mean_absolute_percentage_error(actual, predicted) * 100,
        "r2": r2_score(actual, predicted),
    }


def write_predictions(
    path: Path,
    dates: list[str],
    actual: list[float],
    predicted: list[float],
    baseline: list[float],
    horizon: int = 10,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["market_date", "actual_brent_future", "predicted_brent_future", "baseline_previous_brent", "horizon_days"])
        for d, a, p, b in zip(dates, actual, predicted, baseline):
            writer.writerow([d, a, p, b, horizon])



def main() -> None:
    parser = argparse.ArgumentParser(description="Train a Brent price model for a configurable horizon ahead.")
    parser.add_argument("--market-csv", type=Path, default=Path("datasets") / "ops_market_daily.csv")
    parser.add_argument("--output-dir", type=Path, default=Path("model_artifacts"))
    parser.add_argument("--test-ratio", type=float, default=0.2)
    parser.add_argument("--alpha", type=float, default=0.1)
    parser.add_argument(
        "--horizon",
        type=int,
        default=10,
        help="Number of trading days ahead to predict (default: 10 ≈ 2 calendar weeks).",
    )
    args = parser.parse_args([])

    rows = read_market_rows(args.market_csv)
    x_rows, y_rows, dates = build_examples(rows, horizon=args.horizon)
    if len(x_rows) < 100:
        raise SystemExit("Not enough model-ready rows to train.")

    train_x, train_y, train_dates, test_x, test_y, test_dates = split_chronological(
        x_rows, y_rows, dates, args.test_ratio
    )
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("regressor", Ridge(alpha=args.alpha)),
        ]
    )
    model.fit(train_x, train_y)

    test_predictions = model.predict(test_x).tolist()
    train_predictions = model.predict(train_x).tolist()
    baseline = [row[0] for row in test_x]

    train_metrics = metrics(train_y, train_predictions)
    test_metrics = metrics(test_y, test_predictions)
    baseline_metrics = metrics(test_y, baseline)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    model_path = args.output_dir / "oil_price_model.joblib"
    joblib.dump(model, model_path)
    artifact = {
        "model_type": "sklearn.pipeline.Pipeline(StandardScaler, Ridge)",
        "target": f"{args.horizon}_trading_day_ahead_brent_price_usd",
        "horizon_trading_days": args.horizon,
        "trained_at": date.today().isoformat(),
        "source_file": str(args.market_csv),
        "feature_columns": FEATURE_COLUMNS,
        "model_file": str(model_path),
        "alpha": args.alpha,
        "train_rows": len(train_x),
        "test_rows": len(test_x),
        "train_date_range": [train_dates[0], train_dates[-1]],
        "test_date_range": [test_dates[0], test_dates[-1]],
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "baseline_previous_price_metrics": baseline_metrics,
    }
    (args.output_dir / "oil_price_model.json").write_text(json.dumps(artifact, indent=2), encoding="utf-8")
    write_predictions(args.output_dir / "test_predictions.csv", test_dates, test_y, test_predictions, baseline, horizon=args.horizon)

    print(f"scikit-learn model trained: {args.horizon}-trading-day-ahead Brent price (~2 calendar weeks)")
    print(f"Rows: train={len(train_x)} test={len(test_x)}")
    print(f"Test RMSE: {test_metrics['rmse']:.3f} USD")
    print(f"Test MAE:  {test_metrics['mae']:.3f} USD")
    print(f"Baseline RMSE: {baseline_metrics['rmse']:.3f} USD")
    print(f"Saved: {model_path}")
    print(f"Saved: {args.output_dir / 'oil_price_model.json'}")
    print(f"Saved: {args.output_dir / 'test_predictions.csv'}")

In [ ]:
# Execute: train the model and save artifacts
main()

## 4. Forward Price Forecast
Loads the trained model and feeds the last `horizon` rows of actual market data to produce a day-by-day forward forecast. Source: `src/predict_oil_price.py`

In [ ]:
from __future__ import annotations

import argparse
import csv
import json
from datetime import datetime, timedelta
from pathlib import Path

import joblib


def to_float(value: str | None) -> float:
    if value is None or value == "":
        raise ValueError("Missing numeric input.")
    return float(value)


def estimate_future_trading_date(base_date_str: str, trading_days_ahead: int) -> str:
    """Estimate a calendar date by adding N trading days (skips weekends)."""
    dt = datetime.strptime(base_date_str, "%Y-%m-%d")
    added = 0
    while added < trading_days_ahead:
        dt += timedelta(days=1)
        if dt.weekday() < 5:  # Mon-Fri
            added += 1
    return dt.strftime("%Y-%m-%d")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Generate a forward forecast of Brent crude oil prices."
    )
    parser.add_argument("--metadata", type=Path, default=Path("model_artifacts") / "oil_price_model.json")
    parser.add_argument("--model", type=Path, default=None)
    parser.add_argument("--market-csv", type=Path, default=Path("datasets") / "ops_market_daily.csv")
    args = parser.parse_args([])

    # --- Load model and config ---
    artifact = json.loads(args.metadata.read_text(encoding="utf-8"))
    model_path = args.model or Path(artifact["model_file"])
    horizon: int = artifact.get("horizon_trading_days", 5)
    model = joblib.load(model_path)
    feature_columns = artifact["feature_columns"]

    with args.market_csv.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    sorted_rows = sorted(rows, key=lambda r: r["market_date"])
    last_date = sorted_rows[-1]["market_date"]

    # --- Single-point prediction from the latest row ---
    latest = sorted_rows[-1]
    raw_features = [to_float(latest[col]) for col in feature_columns]
    single_pred = model.predict([raw_features])[0]

    print("=" * 60)
    print("  BRENT CRUDE OIL - FORWARD PRICE FORECAST")
    print("=" * 60)
    print(f"  Last data date  : {last_date}")
    print(f"  Current Brent   : ${float(latest['brent_price_usd']):.2f}")
    print(f"  Model horizon   : {horizon} trading days")
    print(f"  Headline pred.  : ${single_pred:.2f}  ({horizon} days out)")
    print("=" * 60)

    # --- Multi-day forward forecast ---
    # The model predicts price at (source_date + horizon).
    # By feeding the last `horizon` rows of actual data, each row produces
    # a prediction for a different future date:
    #   row at D-horizon+1  →  predicts D+1
    #   row at D-horizon+2  →  predicts D+2
    #   ...
    #   row at D            →  predicts D+horizon
    # This gives us `horizon` predicted future prices.
    forecast_source_rows = sorted_rows[-horizon:]
    forecasts = []

    print(f"\n{'Date':<14} {'Days Ahead':<14} {'Predicted ($)':<16} {'Based On'}")
    print("-" * 60)

    for i, row in enumerate(forecast_source_rows):
        try:
            features = [to_float(row[col]) for col in feature_columns]
        except (ValueError, KeyError):
            continue
        pred = model.predict([features])[0]
        days_ahead = i + 1
        future_date = estimate_future_trading_date(last_date, days_ahead)
        source_date = row["market_date"]
        forecasts.append({
            "forecast_date": future_date,
            "predicted_brent_usd": round(pred, 4),
            "trading_days_ahead": days_ahead,
            "source_date": source_date,
        })
        print(f"{future_date:<14} +{days_ahead:<13} ${pred:<15.2f} {source_date}")

    # --- Save forecast CSV ---
    output_dir = Path(artifact["model_file"]).parent
    forecast_path = output_dir / "forward_forecast.csv"
    output_dir.mkdir(parents=True, exist_ok=True)
    with forecast_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["forecast_date", "predicted_brent_usd", "trading_days_ahead", "source_date"],
        )
        writer.writeheader()
        writer.writerows(forecasts)

    print(f"\nSaved {len(forecasts)}-day forward forecast to {forecast_path}")

In [ ]:
# Execute: generate the forward forecast CSV
main()

## 5. Visualizations
Generates four interactive Plotly charts rendered inline:
- Actual vs Predicted (line, zoomable week-by-week)
- Prediction accuracy scatter
- Model vs Baseline vs Actual
- Forward forecast (recent actual + future predictions)

Use the **1W / 2W / 1M … All** buttons or drag the range slider to zoom.
Source: `Visualization/visualize_predictions.py`

In [ ]:
"""visualize_predictions.py
Interactive Plotly visualizations for the Brent oil price model.

Produces four charts — all fully zoomable and pannable with week-by-week
range-selector buttons:

  1. Actual vs Predicted (test set line chart)
  2. Prediction accuracy scatter (actual vs predicted)
  3. Model vs Baseline vs Actual (test set)
  4. Forward forecast — recent actual prices bridged to future predictions

Run standalone:
    python Visualization/visualize_predictions.py

Or import and call create_visualizations() from a notebook.
"""
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------

def _range_buttons() -> dict:
    """Week-by-week through All-time range selector buttons."""
    return dict(
        buttons=[
            dict(count=7,  label="1W",  step="day",   stepmode="backward"),
            dict(count=14, label="2W",  step="day",   stepmode="backward"),
            dict(count=1,  label="1M",  step="month", stepmode="backward"),
            dict(count=3,  label="3M",  step="month", stepmode="backward"),
            dict(count=6,  label="6M",  step="month", stepmode="backward"),
            dict(count=1,  label="1Y",  step="year",  stepmode="backward"),
            dict(step="all", label="All"),
        ],
        bgcolor="#f0f2f6",
        activecolor="#4a90d9",
    )


def _time_xaxis(title: str = "Date") -> dict:
    """Standard date x-axis with range selector and slider."""
    return dict(
        title=title,
        type="date",
        rangeselector=_range_buttons(),
        rangeslider=dict(visible=True, thickness=0.05),
        showgrid=True,
        gridcolor="#e5e5e5",
    )


def _base_layout(**kwargs) -> dict:
    base = dict(
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=80, b=60),
        plot_bgcolor="#fafafa",
    )
    base.update(kwargs)
    return base


# ---------------------------------------------------------------------------
# Chart builders
# ---------------------------------------------------------------------------

def _chart_actual_vs_predicted(
    df: pd.DataFrame,
    actual_col: str,
    predicted_col: str,
    horizon_label: str,
) -> go.Figure:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["market_date"],
        y=df[actual_col],
        name="Actual Brent Price",
        line=dict(color="royalblue", width=2),
        opacity=0.85,
        hovertemplate="%{x|%Y-%m-%d}<br>Actual: $%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=df["market_date"],
        y=df[predicted_col],
        name=f"Predicted ({horizon_label})",
        line=dict(color="darkorange", width=2),
        opacity=0.85,
        hovertemplate="%{x|%Y-%m-%d}<br>Predicted: $%{y:.2f}<extra></extra>",
    ))

    fig.update_layout(**_base_layout(
        title=f"Brent Crude Oil: Actual vs Predicted — {horizon_label} (Test Set)",
        xaxis=_time_xaxis(),
        yaxis=dict(title="Price (USD)", showgrid=True, gridcolor="#e5e5e5"),
    ))
    return fig


def _chart_scatter(
    df: pd.DataFrame,
    actual_col: str,
    predicted_col: str,
    horizon_label: str,
) -> go.Figure:
    min_val = min(df[actual_col].min(), df[predicted_col].min())
    max_val = max(df[actual_col].max(), df[predicted_col].max())

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df[actual_col],
        y=df[predicted_col],
        mode="markers",
        marker=dict(color="seagreen", opacity=0.45, size=6),
        name="Test Samples",
        hovertemplate="Actual: $%{x:.2f}<br>Predicted: $%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(color="crimson", dash="dash", width=1.5),
        name="Perfect Prediction (y = x)",
    ))

    fig.update_layout(**_base_layout(
        title=f"Prediction Accuracy: Actual vs Predicted ({horizon_label})",
        xaxis=dict(title="Actual Price (USD)", showgrid=True, gridcolor="#e5e5e5"),
        yaxis=dict(title="Predicted Price (USD)", showgrid=True, gridcolor="#e5e5e5"),
        hovermode="closest",
    ))
    return fig


def _chart_model_vs_baseline(
    df: pd.DataFrame,
    actual_col: str,
    predicted_col: str,
    horizon_label: str,
) -> go.Figure:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["market_date"],
        y=df[actual_col],
        name="Actual Brent Price",
        line=dict(color="royalblue", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Actual: $%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=df["market_date"],
        y=df[predicted_col],
        name=f"Model Prediction ({horizon_label})",
        line=dict(color="darkorange", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Model: $%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=df["market_date"],
        y=df["baseline_previous_brent"],
        name="Baseline (Current Day Price)",
        line=dict(color="gray", width=1.5, dash="dash"),
        opacity=0.65,
        hovertemplate="%{x|%Y-%m-%d}<br>Baseline: $%{y:.2f}<extra></extra>",
    ))

    fig.update_layout(**_base_layout(
        title=f"Model vs Baseline vs Actual — {horizon_label}",
        xaxis=_time_xaxis(),
        yaxis=dict(title="Price (USD)", showgrid=True, gridcolor="#e5e5e5"),
    ))
    return fig


def _chart_forward_forecast(
    forecast_csv_path: str | None,
    market_csv_path: str | None,
    horizon_label: str,
) -> go.Figure | None:
    if forecast_csv_path is None or market_csv_path is None:
        print("Skipping forward forecast plot (no forecast CSV or market CSV provided).")
        return None

    try:
        fc = pd.read_csv(forecast_csv_path)
        market = pd.read_csv(market_csv_path)
    except FileNotFoundError as exc:
        print(f"Skipping forward forecast plot: {exc}")
        return None

    fc["forecast_date"] = pd.to_datetime(fc["forecast_date"])
    market["market_date"] = pd.to_datetime(market["market_date"])
    market = market.sort_values("market_date")

    recent = market.tail(60).copy()
    last_actual_date = recent["market_date"].iloc[-1]
    last_actual_price = float(recent["brent_price_usd"].iloc[-1])

    fc_sorted = fc.sort_values("forecast_date")

    # Bridge: prepend the last actual point so the forecast line connects cleanly
    bridge_dates = pd.concat(
        [pd.Series([last_actual_date]), fc_sorted["forecast_date"]],
        ignore_index=True,
    )
    bridge_prices = pd.concat(
        [pd.Series([last_actual_price]), fc_sorted["predicted_brent_usd"].astype(float)],
        ignore_index=True,
    )

    fig = go.Figure()

    # Recent actual prices
    fig.add_trace(go.Scatter(
        x=recent["market_date"],
        y=recent["brent_price_usd"].astype(float),
        name="Actual Brent Price (Recent)",
        line=dict(color="royalblue", width=2.5),
        hovertemplate="%{x|%Y-%m-%d}<br>Actual: $%{y:.2f}<extra></extra>",
    ))

    # Bridged forecast line
    fig.add_trace(go.Scatter(
        x=bridge_dates,
        y=bridge_prices,
        name="Forecasted Brent Price",
        line=dict(color="crimson", width=2.5, dash="dash"),
        mode="lines+markers",
        marker=dict(size=7, color="crimson", symbol="circle"),
        hovertemplate="%{x|%Y-%m-%d}<br>Forecast: $%{y:.2f}<extra></extra>",
    ))

    # Shaded forecast window
    fig.add_vrect(
        x0=last_actual_date,
        x1=fc_sorted["forecast_date"].max(),
        fillcolor="crimson",
        opacity=0.06,
        layer="below",
        line_width=0,
        annotation=dict(
            text="Forecast Window",
            font=dict(size=11, color="crimson"),
            align="left",
        ),
        annotation_position="top left",
    )

    fig.update_layout(**_base_layout(
        title=f"Brent Crude Oil — Forward Price Forecast ({horizon_label})",
        xaxis=_time_xaxis(),
        yaxis=dict(title="Price (USD)", showgrid=True, gridcolor="#e5e5e5"),
    ))
    return fig


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def create_visualizations(
    predictions_csv_path: str,
    output_dir: str | None = None,   # kept for API compatibility; no files are written
    forecast_csv_path: str | None = None,
    market_csv_path: str | None = None,
) -> None:
    """
    Generate four interactive Plotly charts for the Brent oil price model.

    Charts display inline in Jupyter or open in the browser when run as a
    standalone script. No PNG files are written to disk.

    Parameters
    ----------
    predictions_csv_path : str
        Path to model_artifacts/test_predictions.csv produced by train_oil_model.py.
    output_dir : str | None
        Ignored — kept for backwards compatibility.
    forecast_csv_path : str | None
        Path to model_artifacts/forward_forecast.csv produced by predict_oil_price.py.
    market_csv_path : str | None
        Path to datasets/ops_market_daily.csv (used for the forward forecast chart).
    """
    print(f"Loading predictions from {predictions_csv_path} ...")
    try:
        df = pd.read_csv(predictions_csv_path)
    except FileNotFoundError:
        print(f"Error: Could not find {predictions_csv_path}. "
              "Ensure the model has been trained first.")
        return

    df["market_date"] = pd.to_datetime(df["market_date"])
    df = df.sort_values("market_date")

    # Horizon label
    horizon_days = int(df["horizon_days"].iloc[0]) if "horizon_days" in df.columns else 5
    calendar_weeks = round(horizon_days / 5)
    week_str = f"~{calendar_weeks} Week{'s' if calendar_weeks != 1 else ''}"
    horizon_label = f"{horizon_days} Trading Days ({week_str}) Ahead"

    # Column name compatibility (old vs new schema)
    actual_col    = "actual_brent_future"    if "actual_brent_future"    in df.columns else "actual_brent_next"
    predicted_col = "predicted_brent_future" if "predicted_brent_future" in df.columns else "predicted_brent_next"

    # 1. Actual vs Predicted line chart
    fig1 = _chart_actual_vs_predicted(df, actual_col, predicted_col, horizon_label)
    fig1.show()

    # 2. Scatter: correlation / accuracy
    fig2 = _chart_scatter(df, actual_col, predicted_col, horizon_label)
    fig2.show()

    # 3. Model vs Baseline vs Actual
    fig3 = _chart_model_vs_baseline(df, actual_col, predicted_col, horizon_label)
    fig3.show()

    # 4. Forward forecast
    fig4 = _chart_forward_forecast(forecast_csv_path, market_csv_path, horizon_label)
    if fig4 is not None:
        fig4.show()


# ---------------------------------------------------------------------------
# Standalone entry point
# ---------------------------------------------------------------------------

In [ ]:
# Paths mirror the defaults used by the visualize_predictions.py __main__ block.
# Charts are rendered inline (Plotly) — no files are written to disk.
_predictions_csv = str(Path("model_artifacts") / "test_predictions.csv")
_forecast_csv    = str(Path("model_artifacts") / "forward_forecast.csv")
_market_csv      = str(Path("datasets")        / "ops_market_daily.csv")

create_visualizations(
    _predictions_csv,
    forecast_csv_path=_forecast_csv,
    market_csv_path=_market_csv,
)